# Team 18 · Two-stage logistic regression
Choose **Runtime → Run all** in Colab and sign in with the account that can open the team's Drive. CPU works. Results, models and patient checkpoints are saved as the notebook runs. If the session disconnects, run all cells again. The last cell copies the prediction files and a short report to a dedicated results folder in the team's Drive.

This is an exploratory follow-up to the single-stage model. The feature, region and mask rules follow [Maia’s staged random forest](https://github.com/maialist/EXSU500-Pancreatic-Segmentation-Team18/blob/0f845c8e90e50926bae24ba311f293e90f35f9f3/models/random_forest.py). Both classifiers here are logistic regression. Stage 1 finds pancreas including tumour. Stage 2 scores tumour inside the dilated region. The final tumour label must also lie inside the original Stage 1 mask, as in Maia’s code.

In [1]:
%pip -q install nibabel
from google.colab import auth, drive
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pathlib import Path
import hashlib, json, shutil, zipfile
from pathlib import PurePosixPath

file_id = '1_DdvTIa8gs2p1Xqtl_6x4fkK_gtMGh6D'
api = build('drive', 'v3', cache_discovery=False)
meta = api.files().get(fileId=file_id, fields='id,name,size,md5Checksum', supportsAllDrives=True).execute()
version = meta['md5Checksum']
cache = Path('/content/team18_data') / version
cache.mkdir(parents=True, exist_ok=True)
archive = cache / 'processed.zip'
if not archive.is_file():
    print('Downloading the shared archive. This can take several minutes.', flush=True)
    tmp = cache / 'download.partial'
    req = api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with tmp.open('wb') as f:
        downloader = MediaIoBaseDownload(f, req, chunksize=32*1024**2)
        done = False
        while not done:
            status, done = downloader.next_chunk(num_retries=5)
            print(f'{status.progress():.0%}', end=' ', flush=True)
    digest = hashlib.md5()
    with tmp.open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024**2), b''):
            digest.update(chunk)
    if digest.hexdigest() != version:
        raise IOError('Incomplete download. Run this cell again.')
    tmp.replace(archive)

DATA = cache / 'cases'
DATA.mkdir(exist_ok=True)
SPLIT = DATA / 'splits.json'
def unpack_cases(archive, folder):
    names = set()
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            p = PurePosixPath(entry.filename)
            if entry.is_dir() or any(part.startswith('.') or part == '__MACOSX' for part in p.parts):
                continue
            if p.suffix != '.npz' and p.name != 'splits.json':
                continue
            if p.name in names:
                raise ValueError(f'Archive contains ambiguous duplicate: {p.name}')
            names.add(p.name)
            dest = folder / p.name
            if dest.exists() and dest.stat().st_size == entry.file_size:
                continue
            tmp = dest.with_suffix('.partial')
            with z.open(entry) as source, tmp.open('wb') as target:
                shutil.copyfileobj(source, target)
            tmp.replace(dest)
    if 'splits.json' not in names:
        raise FileNotFoundError('The archive has no splits.json. Ask the data lead for the split used by the other models before comparing scores.')
unpack_cases(archive, DATA)
print('Loaded team archive:', meta['name'])

Mounted at /content/drive


3% 5% 8% 10% 13% 15% 18% 20% 23% 25% 28% 30% 33% 35% 38% 40% 43% 45% 48% 50% 53% 55% 58% 60% 63% 65% 68% 70% 73% 75% 78% 80% 83% 85% 88% 90% 93% 95% 98% 100% Loaded team archive: processed.zip


In [2]:
import numpy as np, pandas as pd, joblib, sklearn, io, os
from IPython.display import display, Markdown, HTML
from scipy.ndimage import (gaussian_filter, uniform_filter, gaussian_laplace, binary_dilation,
                           generate_binary_structure, label as connected_components)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits

OUT = Path('/content/drive/MyDrive/Team18_Logistic_Results/staged_lr_v1') / version
OUT.mkdir(parents=True, exist_ok=True)
GAUSSIAN_SIGMAS = (1.0, 2.0, 4.0)
LOCAL_WINDOW_SIZES = (5, 9)
DEFAULT_DILATION_ITERATIONS = 3
DEFAULT_MIN_COMPONENT_SIZE = 100
THRESHOLD_GRID = (0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99)
def compute_feature_channels(image: np.ndarray) -> tuple[np.ndarray, ...]:
    """Compute reusable filter and position channels for one volume."""
    if image.ndim != 3:
        raise ValueError(f"Expected a 3D image, got shape {image.shape}")
    image = image.astype(np.float32, copy=False)
    # Keep one array per channel so the same filters can serve sampling and
    # batched prediction without recalculating them for individual voxels.
    channels: list[np.ndarray] = [image]
    # Smoothed intensities provide context at multiple spatial scales.
    for sigma in GAUSSIAN_SIGMAS:
        channels.append(gaussian_filter(image, sigma=sigma).astype(np.float32))
    # Local moments describe regional brightness and texture.
    for size in LOCAL_WINDOW_SIZES:
        mean = uniform_filter(image, size=size).astype(np.float32)
        mean_squared = uniform_filter(image * image, size=size).astype(np.float32)
        variance = np.maximum(mean_squared - mean * mean, 0.0)
        channels.extend((mean, np.sqrt(variance).astype(np.float32)))
    # Derivative responses capture boundaries and local structural changes.
    gradients = np.gradient(image)
    channels.append(
        np.sqrt(sum(gradient * gradient for gradient in gradients)).astype(np.float32)
    )
    for sigma in (1.0, 2.0):
        channels.append(gaussian_laplace(image, sigma=sigma).astype(np.float32))
    # Consistent preprocessing makes normalized coordinates useful as an
    # anatomical position prior.
    coordinates = np.meshgrid(
        *(np.linspace(0.0, 1.0, size, dtype=np.float32) for size in image.shape),
        indexing="ij",
    )
    channels.extend(coordinates)
    return tuple(channels)


def _features_at_coordinates(
    channels: tuple[np.ndarray, ...], coordinates: np.ndarray
) -> np.ndarray:
    """Gather all feature channels at selected voxel coordinates."""
    # Convert volume channels into one feature row per requested voxel.
    if coordinates.ndim != 2 or coordinates.shape[1] != 3:
        raise ValueError("Coordinates must have shape (n_voxels, 3)")
    if coordinates.size and (
        np.any(coordinates < 0)
        or any(np.any(coordinates[:, axis] >= channels[0].shape[axis]) for axis in range(3))
    ):
        raise ValueError("Voxel coordinates are outside the image bounds")
    indices = tuple(coordinates[:, axis] for axis in range(3))
    return np.column_stack([channel[indices] for channel in channels]).astype(
        np.float32, copy=False
    )


def _stage1_labels(label: np.ndarray) -> np.ndarray:
    """Map pancreas and tumour labels to the Stage 1 foreground class."""
    # Stage 1 treats tumour as part of the pancreas foreground.
    return (label > 0).astype(np.uint8)


def _stage2_labels(label: np.ndarray) -> np.ndarray:
    """Map tumour labels to Stage 2 foreground, with other labels as background."""
    # Stage 2 separates tumour from all other voxels in its pancreas ROI.
    return (label == 2).astype(np.uint8)


def _load_splits(path: Path) -> dict[str, list[str]]:
    """Load and validate the shared, non-overlapping patient split."""
    # All models must use the same non-overlapping patient-level split.
    with path.open(encoding="utf-8") as split_file:
        splits = json.load(split_file)
    required = ("train", "val", "test")
    if not all(isinstance(splits.get(name), list) for name in required):
        raise ValueError(f"{path} must contain train, val, and test case lists")
    if any(not splits[name] for name in required):
        raise ValueError("Train, validation, and test splits must all be non-empty")
    if any(not isinstance(case, str) for name in required for case in splits[name]):
        raise ValueError("Split entries must be case-name strings")
    if len(set(splits["train"] + splits["val"] + splits["test"])) != sum(
        len(splits[name]) for name in required
    ):
        raise ValueError("Patient cases overlap between train, validation, and test")
    return splits


def _largest_component(mask: np.ndarray, min_size: int) -> np.ndarray:
    """Return the largest 3D foreground component if it meets the size limit."""
    components, count = connected_components(mask, structure=generate_binary_structure(3, 1))
    if count == 0:
        return np.zeros(mask.shape, dtype=bool)
    sizes = np.bincount(components.ravel())
    sizes[0] = 0
    largest = int(np.argmax(sizes))
    if sizes[largest] < min_size:
        return np.zeros(mask.shape, dtype=bool)
    return components == largest



splits = _load_splits(SPLIT)
print('Patients:', {k:len(v) for k,v in splits.items()})

def save_arrays(path, **arrays):
    tmp = path.with_suffix('.partial.npz')
    np.savez_compressed(tmp, **arrays)
    tmp.replace(path)

def read_case(name):
    with np.load(DATA / (name+'.npz')) as f:
        x,y = f['image'].astype(np.float32), f['label']
    if x.shape != y.shape or x.ndim != 3 or not np.isfinite(x).all() or not np.isin(y,[0,1,2]).all():
        raise ValueError('Check the prepared arrays for '+name)
    return x,y

def gather(channels, indices):
    coords = np.column_stack(np.unravel_index(indices, channels[0].shape))
    return _features_at_coordinates(channels, coords)

Patients: {'test': 42, 'val': 42, 'train': 197}


## Train both classifiers
Stage 1 samples five background voxels per foreground voxel when available. Stage 2 samples tumour and non-tumour equally within the dilated training reference. The features include normalized intensity, smoothed intensities, local texture, derivative responses and normalized position. Each scaler learns from training voxels only.

In [3]:
def train_stages():
    path = OUT / 'staged_models.joblib'
    if path.exists(): return joblib.load(path)
    folder = OUT / 'training_samples'
    folder.mkdir(exist_ok=True)
    parts = [[], []]
    for i,name in enumerate(splits['train']):
        file = folder / (name+'.npz')
        if not file.exists():
            x,truth = read_case(name)
            channels = compute_feature_channels(x)
            fg = truth > 0
            region = binary_dilation(fg,iterations=3)
            arrays = {}
            for stage,(target,roi,neg_ratio) in enumerate([(fg,None,5),(truth==2,region,1)]):
                rng = np.random.default_rng([42,i,stage])
                for k,limit in [(0,250*neg_ratio),(1,250)]:
                    mask = target == k
                    if roi is not None: mask &= roi
                    ids = np.flatnonzero(mask)
                    take = rng.choice(ids,min(limit,len(ids)),replace=False)
                    arrays[f's{stage}c{k}'] = gather(channels,take)
            save_arrays(file,**arrays)
            del channels
        with np.load(file) as z:
            for stage in (0,1):
                parts[stage].append([z[f's{stage}c{k}'] for k in (0,1)])
        if (i+1)%10==0: print('Training cases prepared:',i+1,flush=True)
    models = []
    for stage,ratio in enumerate([5,1]):
        X,y = [],[]
        rng=np.random.default_rng(42+stage)
        for k,limit in [(0,10000*ratio),(1,10000)]:
            a=np.concatenate([pair[k] for pair in parts[stage]])
            if not len(a): raise ValueError('Training class missing')
            x=a[rng.choice(len(a),min(limit,len(a)),replace=False)]
            X.append(x); y.append(np.full(len(x),k))
        clf=make_pipeline(StandardScaler(),LogisticRegression(C=1,solver='lbfgs',max_iter=1000,random_state=42))
        clf.fit(np.concatenate(X),np.concatenate(y))
        if clf[-1].n_iter_.max()>=1000: raise RuntimeError('Training did not converge')
        models.append(clf)
    tmp=path.with_suffix('.partial.joblib')
    joblib.dump(models,tmp); tmp.replace(path)
    return models
models=train_stages()

Training cases prepared: 10
Training cases prepared: 20
Training cases prepared: 30
Training cases prepared: 40
Training cases prepared: 50
Training cases prepared: 60
Training cases prepared: 70
Training cases prepared: 80
Training cases prepared: 90
Training cases prepared: 100
Training cases prepared: 110
Training cases prepared: 120
Training cases prepared: 130
Training cases prepared: 140
Training cases prepared: 150
Training cases prepared: 160
Training cases prepared: 170
Training cases prepared: 180
Training cases prepared: 190


## Select the two thresholds on validation patients
First choose the Stage 1 threshold by mean Dice for pancreas including tumour. For each validation patient, rebuild the region from that Stage 1 prediction. Choose the Stage 2 threshold by mean tumour Dice within the full-volume evaluation. Ties use the lower threshold. Keep both thresholds fixed for testing.

In [4]:
def probability(clf, channels, indices=None):
    n=channels[0].size if indices is None else len(indices)
    prob=np.empty(n,dtype=np.float32)
    with threadpool_limits(limits=1):
        for start in range(0,n,100000):
            stop=min(n,start+100000)
            ids=np.arange(start,stop) if indices is None else indices[start:stop]
            prob[start:stop]=clf.predict_proba(gather(channels,ids))[:,1]
    return prob.reshape(channels[0].shape) if indices is None else prob

def gate(p,t):
    return _largest_component(p>=t,DEFAULT_MIN_COMPONENT_SIZE)

def stage2_inputs(channels,mask):
    ids=np.flatnonzero(binary_dilation(mask,iterations=DEFAULT_DILATION_ITERATIONS))
    return ids,probability(models[1],channels,ids),mask.ravel()[ids]

def overlap(a,b):
    den=int(a.sum()+b.sum())
    return 2*np.count_nonzero(a & b)/den if den else np.nan

def thresholds(stage,t1=None):
    folder=OUT / ('validation_s1' if stage==1 else f'validation_s2_{t1:.2f}')
    folder.mkdir(exist_ok=True)
    rows=[]
    for i,name in enumerate(splits['val'],1):
        path=folder/(name+'.npz')
        if not path.exists():
            x,y=read_case(name)
            channels=compute_feature_channels(x)
            p=probability(models[0],channels)
            if stage==1:
                values=[overlap(gate(p,t),y>0) for t in THRESHOLD_GRID]
            else:
                ids,q,inside=stage2_inputs(channels,gate(p,t1))
                tumour=y==2
                local=tumour.ravel()[ids]
                values=[]
                for t in THRESHOLD_GRID:
                    called=(q>=t)&inside
                    den=called.sum()+tumour.sum()
                    values.append(2*np.count_nonzero(called&local)/den if den else np.nan)
            save_arrays(path,values=values)
            del channels
        with np.load(path) as z: rows.append(z['values'])
        print(f'Validation stage {stage}: {i}/{len(splits["val"])}',flush=True)
    df=pd.DataFrame(rows,index=splits['val'],columns=THRESHOLD_GRID)
    df.to_csv(OUT/f'val_s{stage}_by_case.csv',index_label='case')
    means=df.mean()
    if means.isna().all(): raise ValueError('No defined validation Dice')
    display(means.rename('mean validation Dice').to_frame())
    return float(means.idxmax())
t1=thresholds(1)
t2=thresholds(2,t1)
print('Selected thresholds:',t1,t2)

Validation stage 1: 1/42
Validation stage 1: 2/42
Validation stage 1: 3/42
Validation stage 1: 4/42
Validation stage 1: 5/42
Validation stage 1: 6/42
Validation stage 1: 7/42
Validation stage 1: 8/42
Validation stage 1: 9/42
Validation stage 1: 10/42
Validation stage 1: 11/42
Validation stage 1: 12/42
Validation stage 1: 13/42
Validation stage 1: 14/42
Validation stage 1: 15/42
Validation stage 1: 16/42
Validation stage 1: 17/42
Validation stage 1: 18/42
Validation stage 1: 19/42
Validation stage 1: 20/42
Validation stage 1: 21/42
Validation stage 1: 22/42
Validation stage 1: 23/42
Validation stage 1: 24/42
Validation stage 1: 25/42
Validation stage 1: 26/42
Validation stage 1: 27/42
Validation stage 1: 28/42
Validation stage 1: 29/42
Validation stage 1: 30/42
Validation stage 1: 31/42
Validation stage 1: 32/42
Validation stage 1: 33/42
Validation stage 1: 34/42
Validation stage 1: 35/42
Validation stage 1: 36/42
Validation stage 1: 37/42
Validation stage 1: 38/42
Validation stage 1: 3

,mean validation Dice
0.50,0.019911
0.60,0.013162
0.70,0.006247
0.80,0.000834
0.90,0.000000
0.95,0.000000
0.98,0.000000
0.99,0.000000


Validation stage 2: 1/42
Validation stage 2: 2/42
Validation stage 2: 3/42
Validation stage 2: 4/42
Validation stage 2: 5/42
Validation stage 2: 6/42
Validation stage 2: 7/42
Validation stage 2: 8/42
Validation stage 2: 9/42
Validation stage 2: 10/42
Validation stage 2: 11/42
Validation stage 2: 12/42
Validation stage 2: 13/42
Validation stage 2: 14/42
Validation stage 2: 15/42
Validation stage 2: 16/42
Validation stage 2: 17/42
Validation stage 2: 18/42
Validation stage 2: 19/42
Validation stage 2: 20/42
Validation stage 2: 21/42
Validation stage 2: 22/42
Validation stage 2: 23/42
Validation stage 2: 24/42
Validation stage 2: 25/42
Validation stage 2: 26/42
Validation stage 2: 27/42
Validation stage 2: 28/42
Validation stage 2: 29/42
Validation stage 2: 30/42
Validation stage 2: 31/42
Validation stage 2: 32/42
Validation stage 2: 33/42
Validation stage 2: 34/42
Validation stage 2: 35/42
Validation stage 2: 36/42
Validation stage 2: 37/42
Validation stage 2: 38/42
Validation stage 2: 3

,mean validation Dice
0.50,0.001427
0.60,0.001348
0.70,0.001111
0.80,0.000476
0.90,0.000013
0.95,0.000000
0.98,0.000000
0.99,0.000000


Selected thresholds: 0.5 0.5


## Test patients and paired comparison
Save a three-label NPZ per test patient. Dice is calculated on the whole volume per patient and class. Compare with the earlier tuned single-stage run by case ID. False-negative tumour voxels are reference tumour voxels missed by the prediction. A completely missed patient has reference tumour and zero overlapping tumour voxels.

In [5]:
def test_cases():
    folder=OUT / f'test_predictions_{t1:.2f}_{t2:.2f}'
    folder.mkdir(exist_ok=True)
    records=[]
    for i,name in enumerate(splits['test'],1):
        x,y=read_case(name)
        path=folder/(name+'.npz')
        if path.exists():
            with np.load(path) as f: pred=f['prediction']
        else:
            channels=compute_feature_channels(x)
            mask=gate(probability(models[0],channels),t1)
            ids,q,inside=stage2_inputs(channels,mask)
            pred=mask.astype(np.uint8)
            pred.ravel()[ids[(q>=t2)&inside]]=2
            save_arrays(path,prediction=pred)
            del channels
        tumour=y==2
        hit=int(np.count_nonzero(tumour & (pred==2)))
        ref=int(tumour.sum()); called=int((pred==2).sum())
        lost=int(np.count_nonzero(tumour & (pred==0)))
        records.append(dict(case=name,tumour=overlap(tumour,pred==2),pancreas=overlap(y==1,pred==1),
            reference=ref,called=called,hit=hit,false_negative=ref-hit,false_positive=called-hit,
            recall=hit/ref if ref else np.nan,missed=int(ref>0 and hit==0),
            missed_by_stage1=lost,missed_inside_stage1=ref-hit-lost))
        print(f'Test {i}/{len(splits["test"])}',flush=True)
    return pd.DataFrame(records).set_index('case'),folder
staged,pred_folder=test_cases()
staged.to_csv(OUT/'staged_test_by_case.csv')

Test 1/42
Test 2/42
Test 3/42
Test 4/42
Test 5/42
Test 6/42
Test 7/42
Test 8/42
Test 9/42
Test 10/42
Test 11/42
Test 12/42
Test 13/42
Test 14/42
Test 15/42
Test 16/42
Test 17/42
Test 18/42
Test 19/42
Test 20/42
Test 21/42
Test 22/42
Test 23/42
Test 24/42
Test 25/42
Test 26/42
Test 27/42
Test 28/42
Test 29/42
Test 30/42
Test 31/42
Test 32/42
Test 33/42
Test 34/42
Test 35/42
Test 36/42
Test 37/42
Test 38/42
Test 39/42
Test 40/42
Test 41/42
Test 42/42


In [6]:
# Snapshot of the previous tuned model's test_by_case.csv in the shared results folder.
baseline=pd.read_csv(io.StringIO("""tumour_overlap_voxels,case,tumour,pancreas,inference_seconds,reference_tumour_voxels,predicted_tumour_voxels
702,pancreas_290,0.002281928717357774,0.020686520081936288,11.707645033000063,1309,613960
1116,pancreas_012,0.003027520322501448,0.026063754619012773,8.389093022999987,2305,734932
1008,pancreas_087,0.0022213750046829583,0.019407170948328643,10.339878163999856,1840,905706
244,pancreas_175,0.0006545503319696868,0.015960717498528815,7.722073404999719,501,745049
256,pancreas_308,0.0006373914753869788,0.02891037054928019,16.62462280999989,356,802918
4310,pancreas_385,0.010938350593170764,0.020294091114118177,9.772413861999667,8036,780017
447,pancreas_379,0.0007130591113241603,0.00704284827766007,14.335914214999775,1287,1252466
542,pancreas_019,0.001571287760190178,0.02359812614040328,10.663561763999951,1107,688773
84281,pancreas_415,0.11135002374813632,0.021176781460505768,14.539756161999776,129870,1383933
617,pancreas_089,0.001705366792058574,0.021137661580268207,8.42459691699969,1019,722579
1460,pancreas_160,0.005794547568869785,0.02085908337432671,7.121651598999961,2802,501120
815,pancreas_126,0.0039036680109973275,0.06405992873425378,6.1099288529999285,1304,416252
1021,pancreas_333,0.0023842411236057385,0.016815457121296257,13.741707670999858,2015,854442
212,pancreas_218,0.0009140629884254618,0.020464041940255896,8.448931183999775,400,463463
639,pancreas_268,0.00315529845716275,0.01653081980833417,5.175361826999961,1054,403979
182,pancreas_418,0.0003990059927630836,0.01734867872485913,7.138623719999941,486,911781
150,pancreas_382,0.0004553962174790176,0.02698547685296036,11.641621574000055,442,658325
313,pancreas_196,0.0012431364372027444,0.02068952030819172,5.0760927619999165,547,503018
390,pancreas_375,0.0007121622707385306,0.023416073542740685,14.630481565000082,967,1094289
629,pancreas_315,0.0020110817317683773,0.02343320247197172,10.882883347000188,1040,624494
433,pancreas_207,0.0017734708421733823,0.03466593967246315,5.893759936000151,627,487681
265,pancreas_235,0.0009499110130532112,0.026217060642228368,8.471346240999992,484,557463
254,pancreas_102,0.0006238747534589434,0.020604123868218825,8.047078517000045,612,813654
3865,pancreas_004,0.008898282159577395,0.02008912764485106,9.545418903000154,6953,861754
514,pancreas_255,0.001364642226119093,0.022156624413960758,14.01934271600021,828,752483
469,pancreas_413,0.0012671463270318246,0.019705222927240043,8.210545374000048,959,739287
844,pancreas_127,0.0014694834086791799,0.023938820861198862,14.720139694999943,1714,1146989
383,pancreas_302,0.0010185465898634668,0.03278250292939599,10.834055164999882,647,751405
126,pancreas_348,0.0004473740739622963,0.015034302381054833,8.434890590000123,294,562993
922,pancreas_412,0.0011920484421594694,0.007914580815715529,25.704925154000193,2323,1544594
456,pancreas_344,0.0011566763817336448,0.01342932950085678,8.87310221000007,706,787760
106,pancreas_321,0.00027328990773887405,0.01608681347157802,14.614794080000138,155,775578
742,pancreas_015,0.0019143669108645483,0.02013733287324661,8.089329776999875,1455,773736
63,pancreas_242,0.0001934004402162401,0.02154341302173188,12.491945035000299,134,651364
56,pancreas_378,0.00022158648005919524,0.02659263946439142,6.8278226799998265,210,505236
905,pancreas_399,0.0016150375161615288,0.01302479363386298,15.729948356000023,2078,1118639
639,pancreas_293,0.0013959630670553808,0.019346896229466948,13.567452251000304,1014,914483
218,pancreas_366,0.000408409488588889,0.022265828862492953,14.754946214000029,858,1066698
2319,pancreas_372,0.005799732897165409,0.01666915300458834,14.11497543500036,4840,794852
608,pancreas_309,0.0014323255619463136,0.01360458219973022,9.83188645600012,1389,847580
70,pancreas_106,0.00022444314053667562,0.029980739599383667,5.5960239799997,391,623375
1083,pancreas_066,0.0010373374430925108,0.013889240065104459,20.54507393999984,2680,2085358""")).set_index('case')
if not baseline.index.is_unique or set(baseline.index)!=set(staged.index):
    raise ValueError('Patient IDs differ from the saved single-stage run; check the archive.')
baseline=baseline.loc[staged.index]
if not np.array_equal(baseline.reference_tumour_voxels,staged.reference):
    raise ValueError('Reference tumour sizes differ across runs.')
baseline['false_negative']=baseline.reference_tumour_voxels-baseline.tumour_overlap_voxels
baseline['false_positive']=baseline.predicted_tumour_voxels-baseline.tumour_overlap_voxels
baseline['recall']=baseline.tumour_overlap_voxels/baseline.reference_tumour_voxels.replace(0,np.nan)
baseline['missed']=((baseline.reference_tumour_voxels>0)&(baseline.tumour_overlap_voxels==0)).astype(int)
metrics=['tumour','pancreas','recall','false_negative','false_positive','missed']
paired=staged[metrics].join(baseline[metrics],lsuffix='_staged',rsuffix='_previous')
paired.to_csv(OUT/'paired_by_case.csv')
def summarize(d):
    return {'mean tumour Dice':d.tumour.mean(),'tumour Dice SD':d.tumour.std(),
            'mean pancreas Dice':d.pancreas.mean(),'pancreas Dice SD':d.pancreas.std(),
            'mean tumour recall':d.recall.mean(),'missed tumour voxels':int(d.false_negative.sum()),
            'extra tumour voxels':int(d.false_positive.sum()),'patients with no tumour overlap':int(d.missed.sum())}
summary=pd.DataFrame({'Previous':summarize(baseline),'Staged':summarize(staged)})
summary.to_csv(OUT/'summary.csv')
display(summary)
# Resample matched patients, keeping both model scores together.
diff=(staged.tumour-baseline.tumour).dropna().to_numpy()
if not len(diff): raise ValueError('No defined paired tumour Dice values')
rng=np.random.default_rng(42)
lo,hi=np.percentile(rng.choice(diff,(10000,len(diff)),replace=True).mean(axis=1),[2.5,97.5])
report=(f'<h1>Two-stage logistic regression</h1><p>We trained two logistic classifiers on {len(splits["train"])} patients. '
    'Stage 1 located pancreas including tumour. Stage 2 learned tumour versus non-tumour in a three-iteration dilation of the reference pancreas. '
    'During validation and testing, the predicted pancreas defined the region. We retained its largest component of at least 100 voxels and restricted final tumour labels to that component.</p>'
    f'<p>We used standardized intensity, texture and spatial features and L2 regularization (C = 1). '
    f'On {len(splits["val"])} validation patients, we selected thresholds {t1:.2f} and {t2:.2f}. '
    f'The table compares both methods on the same {len(staged)} test patients. Recall is the fraction of reference tumour voxels found. Dice measures overlap.</p>'
    +summary.to_html(float_format=lambda x:f'{x:.4f}')+
    f'<p>Mean paired change in tumour Dice (staged minus previous): {diff.mean():+.4f}; '
    f'exploratory 95% patient-bootstrap interval [{lo:+.4f}, {hi:+.4f}], based on {len(diff)} patients. '
    f'Stage 1 excluded {int(staged.missed_by_stage1.sum()):,} tumour voxels. Stage 2 missed another {int(staged.missed_inside_stage1.sum()):,} inside the Stage 1 mask.</p>'
    '<p>This follow-up reused a previously inspected test cohort. Both features and staging changed from the earlier model, so any difference cannot be attributed to staging alone. This is a patient-level internal comparison.</p>')
display(HTML(report))
(OUT/'report.html').write_text(report)

,Previous,Staged
mean tumour Dice,4.541686e-03,3.299691e-03
tumour Dice SD,1.702834e-02,6.804799e-03
mean pancreas Dice,2.153713e-02,1.582846e-02
pancreas Dice SD,8.859091e-03,1.651879e-02
mean tumour recall,5.026269e-01,3.209102e-01
missed tumour voxels,7.536400e+04,1.703440e+05
extra tumour voxels,3.410978e+07,9.497551e+06
patients with no tumour overlap,0.000000e+00,1.500000e+01


,Previous,Staged
mean tumour Dice,0.0045,0.0033
tumour Dice SD,0.0170,0.0068
mean pancreas Dice,0.0215,0.0158
pancreas Dice SD,0.0089,0.0165
mean tumour recall,0.5026,0.3209
missed tumour voxels,75364.0000,170344.0000
extra tumour voxels,34109784.0000,9497551.0000
patients with no tumour overlap,0.0000,15.0000


2207

## Save outputs for the team
The final cell copies all test prediction NPZ files, the score tables and a Google Doc to **Staged logistic regression results** in the shared project folder. If you only have read access there, your personal Drive still holds the results.

In [7]:
from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload
from googleapiclient.errors import HttpError
def find_file(parent,name):
    res=api.files().list(q=f"'{parent}' in parents and trashed=false and name='{name}'",
                         fields='files(id,size,md5Checksum)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
    return next(iter(res.get('files',[])),None)
def ensure_folder(parent,name):
    existing=find_file(parent,name)
    if existing:return existing['id']
    return api.files().create(body={'name':name,'parents':[parent],'mimeType':'application/vnd.google-apps.folder'},
                              fields='id',supportsAllDrives=True).execute()['id']
def copy_output(path,parent):
    found=find_file(parent,path.name)
    digest=hashlib.md5(path.read_bytes()).hexdigest()
    if found and found.get('md5Checksum')==digest:return
    media=MediaFileUpload(str(path),resumable=True)
    if found: api.files().update(fileId=found['id'],media_body=media,supportsAllDrives=True).execute(num_retries=4)
    else: api.files().create(body={'name':path.name,'parents':[parent]},media_body=media,supportsAllDrives=True).execute(num_retries=4)
try:
    dest=ensure_folder('1ayLjdlgDGbSP2yWk8uvZ7EkqzVVDPiQO','Staged logistic regression results')
except HttpError as e:
    if e.resp.status not in (403,404):raise
    dest=ensure_folder('root','Staged logistic regression results')
    print('Team folder unavailable. Saving a shareable results folder in My Drive.')
run_folder=ensure_folder(dest,'Run '+version[:10]+' v1')
pred_dest=ensure_folder(run_folder,'test_predictions')
for name in splits['test']: copy_output(pred_folder/(name+'.npz'),pred_dest)
for f in ['summary.csv','staged_test_by_case.csv','paired_by_case.csv','val_s1_by_case.csv','val_s2_by_case.csv']:
    copy_output(OUT/f,run_folder)
title='Two-stage logistic regression report'
doc=find_file(run_folder,title)
if not doc:
    doc=api.files().create(body={'name':title,'mimeType':'application/vnd.google-apps.document','parents':[run_folder]},
        media_body=MediaIoBaseUpload(io.BytesIO(report.encode()),mimetype='text/html'),fields='id',supportsAllDrives=True).execute()
display(Markdown(f'**[Open Google Doc](https://docs.google.com/document/d/{doc["id"]}/edit)** · '
                 f'**[Open test prediction files](https://drive.google.com/drive/folders/{pred_dest})**'))

**[Open Google Doc](https://docs.google.com/document/d/1x67uIRT6D9QX7r4z7nTAlNH0czSWvYvlwu_i3ypGmVc/edit)** · **[Open test prediction files](https://drive.google.com/drive/folders/1KFl2yATAQ2leclM82JM_d3e7JC0qlTBw)**